# FastFlow — Fabric Anomaly Detection

Pipeline: **Pretrained CNN backbone (ResNet18 / Wide-ResNet50-2) → multi-scale features → Normalizing Flow (per scale) → likelihood-based anomaly map**.

Trained **only on good fabric images**. At test time, low likelihood = anomaly.

## ⚠️ Places you MUST edit (search for `# <-- CHANGE THIS`)
1. **Cell "CONFIG"** — set `train_good_dir` and `test_dir` to your actual Kaggle dataset paths.
2. **Cell "CONFIG"** — confirm the class-folder names inside `test_dir` match your dataset (`good`, `stain`, `holes`, `lines`).
3. (Optional) backbone choice, image size, batch size, epochs — all in CONFIG.

## Expected folder structure
```
<train_good_dir>/
    img001.png
    img002.png
    ...            (1400 good images, flat folder)

<test_dir>/
    good/    (270 images)
    stain/   (300 images)
    holes/   (300 images)
    lines/   (300 images)
```
If your folders are named differently, just fix the `CONFIG` paths / the class list — everything else auto-adapts.


In [ ]:
# Install dependencies (FrEIA = Framework for Easily Invertible Architectures, used to build the normalizing flow)
!pip install -q FrEIA timm scikit-learn


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import timm

import FrEIA.framework as Ff
import FrEIA.modules as Fm

from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


## CONFIG — edit this cell

In [ ]:
CONFIG = {
    # -------- PATHS: CHANGE THESE to match your Kaggle dataset --------
    "train_good_dir": "/kaggle/input/your-dataset-name/train/good",   # <-- CHANGE THIS
    "test_dir":        "/kaggle/input/your-dataset-name/test",        # <-- CHANGE THIS (must contain good/stain/holes/lines subfolders)

    # -------- Class names present inside test_dir --------
    "test_classes": ["good", "stain", "holes", "lines"],              # <-- CHANGE THIS if your folder names differ

    # -------- Model / training settings --------
    "image_size": 256,          # FastFlow default; increase to 384/448 if GPU memory allows and defects are small
    "backbone": "resnet18",     # options: "resnet18" (fast, good baseline) or "wide_resnet50_2" (slower, more accurate)
    "flow_steps": 8,            # number of coupling blocks per scale (paper uses 8)
    "hidden_ratio": 1.0,        # width multiplier for the flow's internal conv subnet
    "conv3x3_only": False,      # if True, all coupling blocks use 3x3 convs; if False, alternates 3x3/1x1 (paper setting)

    "batch_size": 16,
    "val_split": 0.1,           # fraction of good training images held out for validation/loss monitoring
    "epochs": 100,
    "lr": 1e-3,
    "weight_decay": 1e-5,

    "num_workers": 2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "output_dir": "/kaggle/working/fastflow_output",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(CONFIG)


## Dataset

- Training dataset: reads **only** the flat folder of good images.
- Test dataset: reads subfolders, `good` -> label 0, everything else -> label 1 (also keeps the original class name for per-defect-type analysis).


In [ ]:
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

class FabricGoodDataset(Dataset):
    """Flat folder of good (normal) images only — used for training."""
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = [
            os.path.join(root_dir, f)
            for f in sorted(os.listdir(root_dir))
            if f.lower().endswith(IMG_EXTS)
        ]
        if len(self.samples) == 0:
            raise RuntimeError(f"No images found in {root_dir}. Check CONFIG['train_good_dir'].")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, 0, "good", path


class FabricTestDataset(Dataset):
    """root_dir/<class_name>/*.jpg  — good -> label 0, any defect class -> label 1."""
    def __init__(self, root_dir, class_names, transform=None):
        self.transform = transform
        self.samples = []
        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                print(f"WARNING: expected class folder not found: {cls_dir}")
                continue
            label = 0 if cls.lower() == "good" else 1
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(IMG_EXTS):
                    self.samples.append((os.path.join(cls_dir, fname), label, cls))
        if len(self.samples) == 0:
            raise RuntimeError(f"No test images found under {root_dir}. Check CONFIG['test_dir'] / test_classes.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, cls = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, cls, path


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    # Light augmentation is fine (fabric is roughly texture-stationary); avoid heavy geometric distortion
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

full_train_dataset = FabricGoodDataset(CONFIG["train_good_dir"], transform=train_transform)

n_val = int(len(full_train_dataset) * CONFIG["val_split"])
n_train = len(full_train_dataset) - n_val
train_subset, val_subset = torch.utils.data.random_split(
    full_train_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
# validation should use eval_transform (no augmentation) -> wrap
val_subset.dataset.transform = eval_transform if n_val > 0 else train_transform
# NOTE: this changes transform for the whole underlying dataset object; since train/val don't overlap in indices
# and we only need val to be aug-free, this is acceptable for a quick baseline. For strict separation, create
# two separate FabricGoodDataset instances with the same file list split manually.

train_loader = DataLoader(train_subset, batch_size=CONFIG["batch_size"], shuffle=True,
                           num_workers=CONFIG["num_workers"], drop_last=True)
val_loader = DataLoader(val_subset, batch_size=CONFIG["batch_size"], shuffle=False,
                         num_workers=CONFIG["num_workers"]) if n_val > 0 else None

test_dataset = FabricTestDataset(CONFIG["test_dir"], CONFIG["test_classes"], transform=eval_transform)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
                          num_workers=CONFIG["num_workers"])

print(f"Train good images: {n_train} | Val good images: {n_val} | Test images: {len(test_dataset)}")


### Sanity check — visualize a few training samples

In [ ]:
def denormalize(img_tensor):
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = img * std + mean
    return np.clip(img, 0, 1)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    img, label, cls, path = full_train_dataset[i]
    axes[i].imshow(denormalize(img))
    axes[i].set_title(cls)
    axes[i].axis("off")
plt.suptitle("Sample training images (should all be GOOD fabric)")
plt.tight_layout()
plt.show()


## Backbone feature extractor

Pulls multi-scale intermediate features from a frozen, ImageNet-pretrained CNN.


In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self, backbone_name="resnet18"):
        super().__init__()
        if backbone_name not in ["resnet18", "wide_resnet50_2"]:
            raise NotImplementedError(
                f"Backbone '{backbone_name}' is not wired up in this notebook. "
                "Use 'resnet18' or 'wide_resnet50_2', or extend FeatureExtractor for ViT-style backbones (CaiT/DeiT)."
            )
        # out_indices=[1,2,3] grabs 3 mid-level stages -> good multi-scale texture features for FastFlow
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, features_only=True, out_indices=[1, 2, 3]
        )
        self.channels = self.backbone.feature_info.channels()
        self.scales = self.backbone.feature_info.reduction()
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()

    def forward(self, x):
        with torch.no_grad():
            feats = self.backbone(x)
        return feats


## Normalizing flow (per feature scale)

Built with FrEIA. Each scale gets its own invertible flow made of alternating 3x3/1x1 convolutional coupling blocks (`AllInOneBlock`), matching the FastFlow paper.


In [ ]:
def get_subnet_conv_func(kernel_size, hidden_ratio):
    def subnet_conv(in_channels, out_channels):
        hidden_channels = max(1, int(in_channels * hidden_ratio))
        return nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=kernel_size, padding=kernel_size // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, out_channels, kernel_size=kernel_size, padding=kernel_size // 2),
        )
    return subnet_conv


def build_flow(input_chw, flow_steps, conv3x3_only=False, hidden_ratio=1.0):
    """input_chw: (channels, height, width) of the feature map at this scale."""
    nodes = Ff.SequenceINN(*input_chw)
    for i in range(flow_steps):
        kernel_size = 3 if (conv3x3_only or i % 2 == 0) else 1
        nodes.append(
            Fm.AllInOneBlock,
            subnet_constructor=get_subnet_conv_func(kernel_size, hidden_ratio),
            affine_clamping=2.0,
            permute_soft=False,
        )
    return nodes


## FastFlow model

Wraps the frozen backbone + one flow per scale. During training, minimizes negative log-likelihood of normal features under a standard normal prior. During eval, also returns a full-resolution anomaly heatmap.


In [ ]:
class FastFlowModel(nn.Module):
    def __init__(self, backbone_name, input_size, flow_steps, conv3x3_only, hidden_ratio):
        super().__init__()
        self.feature_extractor = FeatureExtractor(backbone_name)
        self.input_size = input_size

        self.norms = nn.ModuleList()
        self.nf_flows = nn.ModuleList()
        for ch, scale in zip(self.feature_extractor.channels, self.feature_extractor.scales):
            fmap_size = input_size // scale
            self.norms.append(nn.LayerNorm([ch, fmap_size, fmap_size], elementwise_affine=True))
            self.nf_flows.append(build_flow((ch, fmap_size, fmap_size), flow_steps, conv3x3_only, hidden_ratio))

    def train(self, mode=True):
        # keep the frozen backbone (and its BatchNorm stats) in eval mode always
        super().train(mode)
        self.feature_extractor.backbone.eval()
        return self

    def forward(self, x):
        features = self.feature_extractor(x)
        loss = 0.0
        z_outputs = []
        for i, feature in enumerate(features):
            feature = self.norms[i](feature)
            z, log_jac_dets = self.nf_flows[i](feature)
            loss = loss + torch.mean(0.5 * torch.sum(z ** 2, dim=(1, 2, 3)) - log_jac_dets)
            z_outputs.append(z)

        ret = {"loss": loss}

        if not self.training:
            anomaly_maps = []
            for z in z_outputs:
                # negative log-likelihood per spatial location -> higher = more anomalous
                log_prob = -torch.mean(z ** 2, dim=1, keepdim=True) * 0.5
                prob = torch.exp(log_prob)
                a_map = F.interpolate(-prob, size=[self.input_size, self.input_size],
                                       mode="bilinear", align_corners=False)
                anomaly_maps.append(a_map)
            anomaly_map = torch.mean(torch.stack(anomaly_maps, dim=-1), dim=-1)
            ret["anomaly_map"] = anomaly_map  # (B, 1, H, W), higher = more anomalous
        return ret


## Build model + optimizer

In [ ]:
device = torch.device(CONFIG["device"])
print("Using device:", device)

model = FastFlowModel(
    backbone_name=CONFIG["backbone"],
    input_size=CONFIG["image_size"],
    flow_steps=CONFIG["flow_steps"],
    conv3x3_only=CONFIG["conv3x3_only"],
    hidden_ratio=CONFIG["hidden_ratio"],
).to(device)

trainable_params = [p for p in model.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
print(f"Trainable parameters (flows only, backbone frozen): {n_trainable:,}")

optimizer = torch.optim.Adam(trainable_params, lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])


## Training loop

Trains only on good images. Tracks train + val loss (val loss should trend down too — it's still all normal/good images, just held-out ones).


In [ ]:
history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
best_ckpt_path = os.path.join(CONFIG["output_dir"], "fastflow_best.pt")

for epoch in range(CONFIG["epochs"]):
    model.train()
    running_loss = 0.0
    for images, labels, cls, paths in train_loader:
        images = images.to(device)
        optimizer.zero_grad()
        ret = model(images)
        loss = ret["loss"]
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / max(1, len(train_loader))
    history["train_loss"].append(train_loss)

    val_loss = None
    if val_loader is not None:
        model.eval()
        running_val = 0.0
        with torch.no_grad():
            for images, labels, cls, paths in val_loader:
                images = images.to(device)
                ret = model(images)
                running_val += ret["loss"].item()
        val_loss = running_val / max(1, len(val_loader))
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_ckpt_path)

    scheduler.step()

    if val_loss is not None:
        print(f"Epoch {epoch+1:3d}/{CONFIG['epochs']} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")
    else:
        print(f"Epoch {epoch+1:3d}/{CONFIG['epochs']} | train_loss={train_loss:.4f}")

# if no val split, just save final weights
if val_loader is None:
    torch.save(model.state_dict(), best_ckpt_path)

print("Best checkpoint saved to:", best_ckpt_path)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["train_loss"], label="train_loss")
if history["val_loss"]:
    plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("negative log-likelihood loss")
plt.title("Training curve")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "training_curve.png"))
plt.show()


## Load best checkpoint and run inference on the full test set

In [ ]:
model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
model.eval()

all_scores = []      # image-level anomaly score (max of the pixel map)
all_labels = []      # 0 = good, 1 = defective
all_maps = []        # per-image anomaly map (H, W)
all_classes = []     # original class name: good / stain / holes / lines
all_paths = []
all_images = []      # keep tensors for visualization

with torch.no_grad():
    for images, labels, cls, paths in test_loader:
        images_gpu = images.to(device)
        ret = model(images_gpu)
        amap = ret["anomaly_map"].squeeze(1).cpu().numpy()  # (B, H, W)
        scores = amap.reshape(amap.shape[0], -1).max(axis=1)

        all_scores.extend(scores.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_maps.extend(list(amap))
        all_classes.extend(list(cls))
        all_paths.extend(list(paths))
        all_images.extend(list(images.cpu()))

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)
print(f"Ran inference on {len(all_scores)} test images.")


## Image-level evaluation

Since no ground-truth defect masks were provided, metrics below are **image-level only** (good vs. defective). Heatmaps further down are qualitative pixel-level visualizations.


In [ ]:
auroc = roc_auc_score(all_labels, all_scores)
print(f"Image-level ROC-AUC: {auroc:.4f}")

fpr, tpr, thresholds = roc_curve(all_labels, all_scores)

# Youden's J statistic to pick an operating threshold
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold = thresholds[best_idx]
print(f"Suggested threshold (Youden's J): {best_threshold:.4f}")

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"ROC (AUC = {auroc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.scatter(fpr[best_idx], tpr[best_idx], color="red", zorder=5, label="chosen threshold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve — image-level anomaly detection")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "roc_curve.png"))
plt.show()


In [ ]:
preds = (all_scores >= best_threshold).astype(int)

print(classification_report(all_labels, preds, target_names=["good", "defective"]))

cm = confusion_matrix(all_labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["good", "defective"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion matrix (image-level)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "confusion_matrix.png"))
plt.show()


In [ ]:
# Per-defect-type breakdown (recall for each defect class separately, using the good/defective threshold)
import pandas as pd

df = pd.DataFrame({"score": all_scores, "label": all_labels, "cls": all_classes})
df["pred"] = preds

rows = []
for cls_name in CONFIG["test_classes"]:
    sub = df[df["cls"] == cls_name]
    if len(sub) == 0:
        continue
    if cls_name.lower() == "good":
        acc = (sub["pred"] == 0).mean()
        rows.append({"class": cls_name, "n": len(sub), "correct_rate (specificity)": acc})
    else:
        acc = (sub["pred"] == 1).mean()
        rows.append({"class": cls_name, "n": len(sub), "correct_rate (recall)": acc})

per_class_df = pd.DataFrame(rows)
print(per_class_df)


## Heatmap visualization

Original image / raw anomaly heatmap / overlay, for one sample of each class.


In [ ]:
def get_first_index_of_class(cls_name):
    for i, c in enumerate(all_classes):
        if c == cls_name:
            return i
    return None

classes_to_show = CONFIG["test_classes"]
fig, axes = plt.subplots(len(classes_to_show), 3, figsize=(12, 4 * len(classes_to_show)))
if len(classes_to_show) == 1:
    axes = axes[None, :]

for row, cls_name in enumerate(classes_to_show):
    idx = get_first_index_of_class(cls_name)
    if idx is None:
        continue
    orig = denormalize(all_images[idx])
    amap = all_maps[idx]
    amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)

    axes[row, 0].imshow(orig)
    axes[row, 0].set_title(f"{cls_name} — original")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(amap_norm, cmap="jet")
    axes[row, 1].set_title("anomaly heatmap")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(orig)
    axes[row, 2].imshow(amap_norm, cmap="jet", alpha=0.5)
    axes[row, 2].set_title("overlay")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "heatmap_results.png"), dpi=150)
plt.show()


In [ ]:
# Optional: grid of several samples per defect class, useful to eyeball failure cases
def show_grid_for_class(cls_name, n=4):
    idxs = [i for i, c in enumerate(all_classes) if c == cls_name][:n]
    fig, axes = plt.subplots(2, len(idxs), figsize=(4 * len(idxs), 8))
    for col, idx in enumerate(idxs):
        orig = denormalize(all_images[idx])
        amap = all_maps[idx]
        amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
        axes[0, col].imshow(orig)
        axes[0, col].set_title(f"{cls_name} #{col} (score={all_scores[idx]:.2f})", fontsize=9)
        axes[0, col].axis("off")
        axes[1, col].imshow(orig)
        axes[1, col].imshow(amap_norm, cmap="jet", alpha=0.5)
        axes[1, col].axis("off")
    plt.tight_layout()
    plt.show()

for cls_name in CONFIG["test_classes"]:
    if cls_name.lower() != "good":
        show_grid_for_class(cls_name, n=4)


## Save everything

Model weights + config + scores are saved to `CONFIG['output_dir']` (Kaggle's `/kaggle/working`), which you can download from the notebook output panel.


In [ ]:
import json as _json

torch.save(model.state_dict(), os.path.join(CONFIG["output_dir"], "fastflow_final.pt"))

with open(os.path.join(CONFIG["output_dir"], "config.json"), "w") as f:
    _json.dump({k: v for k, v in CONFIG.items() if k != "device"}, f, indent=2)

results_df = pd.DataFrame({
    "path": all_paths,
    "class": all_classes,
    "label": all_labels,
    "score": all_scores,
    "pred": preds,
})
results_df.to_csv(os.path.join(CONFIG["output_dir"], "test_results.csv"), index=False)

print("Saved to:", CONFIG["output_dir"])
print(os.listdir(CONFIG["output_dir"]))


## Notes / next steps

- **If AUROC is underwhelming**: try `backbone = "wide_resnet50_2"`, increase `image_size` to 384 (small defects like thin lines/stains benefit from higher resolution), or increase `epochs`.
- **If you get ground-truth defect masks later**: I can extend the evaluation cell to compute pixel-level AUROC/AP and per-pixel Dice/IoU against the masks — that requires the anomaly map (`all_maps`, already saved) and mask images.
- **If you want the CaiT/DeiT backbone from the original diagram**: the `FeatureExtractor` class is the only thing that needs replacing — it would extract per-patch tokens, reshape them into a spatial grid, and feed those into `build_flow` the same way. Happy to build that variant separately once the CNN baseline is confirmed working, since it's a distinct code path from what's here.
